In [2]:
!pip install ddgs trafilatura -q

In [27]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from openai import OpenAI
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura
from pprint import pprint
from IPython.display import Markdown, display



load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
    raise Exception("API Key is missing!")
else:
    print("Key is: " + OPENAI_API_KEY[:8])

client = OpenAI(api_key=OPENAI_API_KEY)

Key is: sk-proj-


In [28]:

def search_web(query):
    ddgs = DDGS()

    results = ddgs.text(query, max_result = 5)
    pprint(f"Got results: {results}")
    return json.dumps(results, indent=2)

In [29]:
def fetch_url(url: str):

    downloaded = trafilatura.fetch_url(url)

    if downloaded:
        text = trafilatura.extract(downloaded)
        print(text)
        if text: 
            print(f" \u2705 Got Text")
            return text
    print(f"\u274C fail to fetch text from URL {url}. ")
    return (f"Could not extract text from url {url}. Try different source.")


In [30]:
search_web("AI in healthcare in 2030")

("Got results: [{'title': 'The Growing Role of Artificial Intelligence in "
 "Healthcare', 'href': "
 "'https://mayomagazine.mayoclinic.org/2026/04/ai-in-healthcare/', 'body': "
 '"Artificial intelligence (AI) is a technological revolution that promises to '
 'redefine the contours of modern medicine. From accelerating diagnoses to '
 "personalizing treatment plans, AI is poised to tackle some of healthcare's "
 'most pressing challenges. The Rise of AI in Healthcare Alan Turing first '
 'coined the term “artificial intelligence” in the 1950s. Two decades later, '
 'AI took its initial ..."}, {\'title\': \'Adoption of artificial intelligence '
 "in healthcare: survey of ...', 'href': "
 "'https://pmc.ncbi.nlm.nih.gov/articles/PMC12202002/', 'body': 'The US "
 'healthcare system faces significant challenges, including clinician burnout, '
 'operational inefficiencies, and concerns about patient safety. Artificial '
 'intelligence (AI), particularly generative AI, has the potential to ad

'[\n  {\n    "title": "The Growing Role of Artificial Intelligence in Healthcare",\n    "href": "https://mayomagazine.mayoclinic.org/2026/04/ai-in-healthcare/",\n    "body": "Artificial intelligence (AI) is a technological revolution that promises to redefine the contours of modern medicine. From accelerating diagnoses to personalizing treatment plans, AI is poised to tackle some of healthcare\'s most pressing challenges. The Rise of AI in Healthcare Alan Turing first coined the term \\u201cartificial intelligence\\u201d in the 1950s. Two decades later, AI took its initial ..."\n  },\n  {\n    "title": "Adoption of artificial intelligence in healthcare: survey of ...",\n    "href": "https://pmc.ncbi.nlm.nih.gov/articles/PMC12202002/",\n    "body": "The US healthcare system faces significant challenges, including clinician burnout, operational inefficiencies, and concerns about patient safety. Artificial intelligence (AI), particularly generative AI, has the potential to address these .

In [31]:
fetch_url("https://www.linkedin.com/pulse/ai-healthcare-2030-opportunities-challenges-strategic-riken-shah-pk86f")

AI in Healthcare 2030: Opportunities, Challenges, and Strategic Insights
Healthcare is entering a transformative era. The volume of patient data, medical literature, and imaging is growing exponentially, creating new pressures for clinicians and administrative staff. Workforce shortages and increasing patient demand are only amplifying these challenges.
We explored these issues in a recent webinar featuring Jan Beger , a healthcare AI veteran with over 20 years of experience. We discussed how AI can help health systems manage these pressures, streamline workflows, and improve patient outcomes.
In practice, we’ve helped hospitals implement AI tools that prioritize critical imaging, optimize scheduling, and identify high-risk patients for follow-up. These solutions reduce administrative burdens, enhance operational efficiency, and allow clinicians to focus on delivering quality care.
At OSP, we focus on integrating AI seamlessly into workflows, ensuring technology enhances human capabili

"AI in Healthcare 2030: Opportunities, Challenges, and Strategic Insights\nHealthcare is entering a transformative era. The volume of patient data, medical literature, and imaging is growing exponentially, creating new pressures for clinicians and administrative staff. Workforce shortages and increasing patient demand are only amplifying these challenges.\nWe explored these issues in a recent webinar featuring Jan Beger , a healthcare AI veteran with over 20 years of experience. We discussed how AI can help health systems manage these pressures, streamline workflows, and improve patient outcomes.\nIn practice, we’ve helped hospitals implement AI tools that prioritize critical imaging, optimize scheduling, and identify high-risk patients for follow-up. These solutions reduce administrative burdens, enhance operational efficiency, and allow clinicians to focus on delivering quality care.\nAt OSP, we focus on integrating AI seamlessly into workflows, ensuring technology enhances human cap

In [32]:
tools = []

search_web_function = {
    "name" : "search_web",
    "description": "search the web using DuckDuckGo and return 3 results",
    "parameters": {
        "type": "object",
        "properties":{
            "query": {
                "type": "string",
                "description": "The search query information on the web"
            }
        },
        "required": ["query"]
    }
}

tools.append({"type":"function", "function": search_web_function} )

#-------------------------------------------

fetch_url_function = {
    "name" : "fetch_url",
    "description": "fetch url the main context from the web page",
    "parameters": {
        "type": "object",
        "properties":{
            "url": {
                "type": "string",
                "description": "fetch url extract text from the web"
            }
        },
        "required": ["url"]
    }
}

tools.append({"type":"function", "function": fetch_url_function} )


In [33]:
tools


[{'type': 'function',
  'function': {'name': 'search_web',
   'description': 'search the web using DuckDuckGo and return 3 results',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The search query information on the web'}},
    'required': ['query']}}},
 {'type': 'function',
  'function': {'name': 'fetch_url',
   'description': 'fetch url the main context from the web page',
   'parameters': {'type': 'object',
    'properties': {'url': {'type': 'string',
      'description': 'fetch url extract text from the web'}},
    'required': ['url']}}}]

In [34]:
def handle_tool_call(tool_calls):
    tools_results = []
    # return what to add to our context (about tool call results, a dictionary)
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        print(f" \U0001f527 Calling function {function_name}")
        args = json.loads(tool_call.function.arguments)

        #Route to function name
        if function_name == "search_web":

            result = search_web(args['query'])
            #print(f"Send notification: {args['message']}")
            content = f"Search Result: {result}"
        elif function_name == "fetch_url":
            result = fetch_url(args['url'])
            content = f"Fetch result: {result}"
        else:
            content = f"Unknown function {function_name}"

        tool_call_result  = {
            "role":"tool",
            "content":content,
            "tool_call_id":tool_call.id
            
        }

        tools_results.append(tool_call_result)

    return tools_results

## Step 4: System Prompt

In [ ]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that "DONE:" should be at the start of the final response, so that
it can be easily parsed and extracted. You CANNOT and SHOULD NOT include "DONE:" 
in any other part of your response except at the very beginning of the final research.
 
Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

## Step 5: Agentic Loop

In [42]:
from json import tool
from urllib import response


def run_research_agent(topic: str, max_iterations: int = 10) -> str:
    """ 
    Run a reseach a topic

    Args: 
        topic: The topic to research
        max_iternation: limit to prevent infinite loop
    Return: 
        The research brief

    """

    print("\n \U0001F50D Staring researching on topic: {topic}")
    print("=" * 60)

    messages = [
        {"role":"system", "content": RESEARCH_AGENT_PROMPT},
        {"role": "user", "content": f"Research the following topic and return a comprehensive research brief: \n\n {topic}"}
    ]

    iteration = 0
    while iteration < max_iterations:
        iteration += 1
        print(f"Iteratin {iteration}")

        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools = tools,
            max_tokens=1000
        )

        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            tool_results = handle_tool_call(message.tool_calls)
            messages.extend(tool_results)
        else:
            content = message.content
            if content.startswith("DONE"):
                brief = content[len("DONE:"):].strip()
                print("\n \u2705 Research complete: \n{brief}")

                return brief
            else:

                print(f" \U0001F4AD Agent is thinking: ")
                pprint(content)

        if iteration == max_iterations - 1:
            print(" \u26A0 Stop researching in next iteration")
            messages.append({"role": "user", "content":"YOu have reached max iterations. Please deliver your brief with DONE"})
    
    #fall back return

    return "Research incomplete. Max iterations reached without finalizing brief."



In [47]:
brief = run_research_agent("Search hotels in cocoa beach with price less than $150.00 per day and only 5 mins walking to the beach. Use travel websites to search")
display(Markdown(brief))


 🔍 Staring researching on topic: {topic}
Iteratin 1
 🔧 Calling function search_web
("Got results: [{'title': 'HILTON COCOA BEACH OCEANFRONT - Updated 2026 Prices "
 "& Hotel', 'href': "
 "'https://www.tripadvisor.com/Hotel_Review-g34145-d84283-Reviews-Hilton_Cocoa_Beach_Oceanfront-Cocoa_Beach_Brevard_County_Florida.html', "
 "'body': '... located near our Cocoa Beach hotel, visit Kennedy Space Center, "
 'The Brevard Zoo, Cocoa Beach Pier, Space Coast Stadium, or simply relax and '
 "unwind in ...'}, {'title': 'THE 10 BEST Hotels near 32922 (Cocoa, FL)', "
 "'href': "
 "'https://www.tripadvisor.com/HotelsNear-g34144-d18868627-32922-Cocoa_Brevard_County_Florida.html', "
 "'body': 'Paddy s during the same week it s difficult to find a hotel room in "
 'the Cocoa Beach area on a tight budget. ... the Homewood Suites by Hilton '
 "Cape ...'}, {'title': 'THE BEST Cocoa Beach Motels With Indoor Pools 2026 - "
 "Tripadvisor', 'href': "
 "'https://www.tripadvisor.com/HotelsList-Cocoa_Beach-Mo

Research Brief on Hotels in Cocoa Beach Under $150 with Beach Proximity

Key Facts and Statistics:
- Average hotel prices in Cocoa Beach fluctuate seasonally, with the cheapest month being October (average rate around $187/night) and the most expensive month March (around $323/night).
- The lowest hotel rates are typically found by booking well in advance (at least 85 days prior).
- Some budget hotels near the beach offer rooms under $150 per night, particularly in off-peak seasons or through good deals.
- Walking distance to the beach is generally considered within a 5-minute walk or up to roughly 0.3 miles.

Main Themes and Arguments from Sources:
- Budget hotels within walking distance to the beach in Cocoa Beach typically include well-known names such as La Quinta Inn & Suites Cocoa Beach, Days Inn by Wyndham Cocoa Beach Port Canaveral, and Motel 6, all offering affordable amenities and proximity to local attractions like the Cocoa Beach Pier.
- Affordable options for families and groups are also available, with some hotels offering suites that can accommodate larger parties.
- Many budget-friendly hotels provide free beach access, free WiFi, and sometimes additional amenities such as pools or health centers.
- The location near the beach is a high priority, and many hotels successfully combine affordability with closeness to the shore.
- The local area also provides convenient access to the Kennedy Space Center, local dining, shopping, and entertainment within walking or short driving distance.
- It is often recommended to stay south of Cocoa Beach or near South Cocoa Beach for cheaper rates, though they are slightly farther from major piers.

Notable Data Points:
- La Quinta Inn & Suites Cocoa Beach-Port Canaveral: Approximately a 4-minute walk to the beach, rooms frequently priced under $150 especially outside peak travel periods.
- Days Inn by Wyndham Cocoa Beach Port Canaveral: About 3-minute walk to the beach and pier, offering spacious and comfortable rooms within the budget.
- Motel 6 Cocoa Beach: Known for extended stays with laundry facilities, heating pool, and budget rates.
- Beachside Hotel & Suites offers affordable suites with amenities such as lazy river pools, free use of beach gear (wagons, umbrellas, surfboards), and proximity to the beach.
- Some lodging options provide free shuttle service to Melbourne Orlando International Airport and have EV charging stations.
- Booking on weekdays like Monday often yields cheaper prices than weekends, and October is ideal for lower prices.

Summary:
Travelers searching for hotels in Cocoa Beach for less than $150 per night within a 5-minute walking distance to the beach have several options. Affordable brands like La Quinta Inn, Days Inn, and Motel 6 are popular choices with close proximity to beaches and other local attractions. Seasonal trends influence prices significantly, with cheaper rates generally available in the fall months and mid-week bookings. These hotels offer a balance of beach access, comfort, and budget-friendly prices, making Cocoa Beach a welcoming destination for tourists and families on a budget.

Sources:
- KAYAK: https://www.kayak.com/Cocoa-Beach-Hotels.16591.hotel.ksp
- Booking.com: https://www.booking.com/budget/city/us/cocoa-beach.html
- Hotels.com: https://www.hotels.com/de1441551-qu0/cheap-hotels-cocoa-beach-florida/
- TripAdvisor (general results): https://www.tripadvisor.com/HotelsList-Cocoa_Beach-Cheap-Hotels-zfp10955.html (summary based on search results)

This brief encapsulates current options and pricing trends for budget hotel stays in Cocoa Beach with emphasis on beach proximity and affordability.

In [ ]:
JUDGE_PROMPT = """# Judge Prompt: Insufficient Research Breadth (TRUE/FALSE)

You are scoring a research agent's deliverable research brief to determine 
if there was a
research breadth failure. Return only TRUE or FALSE.

Definitions

Source:
Any distinctly identifiable origin of information referenced in the brief. This
includes named websites, publications, reports, databases, articles, papers, 
organizations, or any other distinguishable information source. Multiple references to the same 
source count as one source.

Research breadth failure (label TRUE):
The research brief references fewer than 6
distinct sources. When counting:

1. Distinct sources only:
If the same website, publication, or organization is referenced
multiple times, it counts as one source.

2. Identifiable references required:
A source must be specific enough to be
independently locatable — vague attributions like "studies show," "experts say," or
"according to reports" do NOT count as sources.

3. Embedded or inline references count:
Sources do not need to appear in a formal
bibliography. References woven into the body text (e.g., "according to a McKinsey 2024
report" or "data from the Bureau of Labor Statistics") count as valid sources.

4. Agent's own reasoning is not a source:
The agent's synthesis, analysis, or opinions do
not count toward the source total.

No research breadth failure (label FALSE):
The research brief references 6 or more
distinct, identifiable sources anywhere in the deliverable — whether in a bibliography,
inline citations, or body-text attributions.

Output Format

Return exactly one token: TRUE or FALSE. No explanations.

"""

In [ ]:
TOPICS = [
    "The impact of generative AI on pharmaceutical drug discovery"
    "Comparative analysis of nuclear fusion startups and their commercial viability"
    "How central bank digital currencies are reshaping cross-border payments"
    "The state of microplastics regulation in the European Union"
    "Remote work's long-term effects on commercial real estate valuations"
    "Advances in solid-state battery technology for electric vehicles"
    "The role of satellite internet constellations in bridging the global digital divide"
    "Youth mental health trends and social media platform accountability"
    "Water scarcity solutions in the Middle East: desalination vs. conservation"
    "The economic impact of sports stadium subsidies on local communities"
]

In [ ]:
JUDGE_MODEL = "gpt-4.1"
MODEL = "gpt-4.1-nano"


In [ ]:
import contextlib
import io


results = []

for i, topic in enumerate(TOPICS):
    # 1. Run agent
    # suppress printout from the agent for cleaner eval logs  
    with contextlib.redirect_stdout(io.StringIO()):
        brief = run_research_agent(topic, max_iterations=10)

    # 2. Judge
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": brief},
        ],
        max_tokens=1,
        temperature=0,
    )
    verdict = response.choices[0].message.content.strip()
    failed = verdict == "TRUE"

    results.append({"topic": topic, "verdict": verdict, "failed": failed})
    print(f"  → {'FAIL' if failed else 'PASS'}")

# --- Summary ---
n_fail = sum(r["failed"] for r in results)
print(f"\n{'='*50}")
print(f"Results: {len(results) - n_fail}/{len(results)} passed  |  {n_fail}/{len(results)} failed")
print(f"{'='*50}")
for r in results:
    status = "FAIL ✗" if r["failed"] else "PASS ✓"
    print(f"  [{status}] {r['topic'][:65]}")